# Demo: Zero-Shot Forecasting with a Foundation Model


In [ ]:
import os
import warnings
os.environ["HF_HOME"] = os.path.expanduser("~/.cache/huggingface")
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape

HORIZON = 12

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

# Udacity brand colors
UB = {
    "Brand Blue": "#175CFF", "Medium Blue": "#7BA2FF", "Light Blue": "#D2DFFF", "Navy": "#031643"
}
UN = {
    "Black": "#0A0B0F", "Dark Gray": "#5A5C62",
    "Medium Gray": "#9C9FA8", "Gray": "#CECFD4",
    "Light Gray": "#F2F2F2", "White": "#FFFFFF"
}
US = {
    "Orange": "#EE7622", "Yellow": "#F9DC5C",
    "Red": "#9C0D22", "Green": "#23CE6B"
}

mpl.rcParams["lines.linewidth"] = 3
mpl.rcParams["axes.linewidth"] = 2


In [ ]:
df = pd.read_csv("../data/subscribers.csv", parse_dates=["date"], index_col="date").asfreq("MS")
subscribers = df["subscribers"]
ts = TimeSeries.from_series(subscribers)
train, test = ts.split_after(pd.Timestamp("2023-12-01"))
print(f"Train: {len(train)} months, Test: {len(test)} months")

## Running Chronos-2


In [ ]:
try:
    from darts.models import Chronos2Model
    chronos = Chronos2Model(model_name="amazon/chronos-bolt-small")
    chronos_fc = chronos.predict(HORIZON, series=train, num_samples=100)
    print("Chronos-2 forecast generated.")
except ImportError:
    print("Chronos-2 not available. Using ARIMA fallback.")
    fallback = DartsARIMA(p=2, d=1, q=1)
    fallback.fit(train)
    chronos_fc = fallback.predict(HORIZON)

## ARIMA Baseline


In [ ]:
arima = DartsARIMA(p=2, d=1, q=1)
arima.fit(train)
arima_fc = arima.predict(HORIZON)

## Comparing


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ts.plot(ax=ax, label="Actual", color=UN["Black"])
arima_fc.plot(ax=ax, label="ARIMA(2,1,1)", color=UB["Brand Blue"])
if chronos_fc.n_samples > 1:
    chronos_fc.plot(ax=ax, label="Chronos-2", color=US["Green"], low_quantile=0.05, high_quantile=0.95)
else:
    chronos_fc.plot(ax=ax, label="Chronos-2 (fallback)", color=US["Green"])
ax.set_ylabel("Subscribers", fontsize=16)
ax.set_title("Forecast Comparison", fontsize=18, fontweight="bold")
ax.tick_params(axis="both", labelsize=14)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Backtesting


In [ ]:
full = TimeSeries.from_series(subscribers)
bt = arima.historical_forecasts(full, start=0.75, forecast_horizon=12, stride=1, retrain=False)

print(f"MAE:  {mae(full, bt):,.0f}")
print(f"RMSE: {rmse(full, bt):,.0f}")
print(f"MAPE: {mape(full, bt):.1f}%")